# 01 — BERTopic Pipeline

This notebook applies BERTopic with the all-MiniLM-L6-v2 model on merged Nursing transcripts
(Evaluation, Practice, and Education).  

**Inputs:**  
- `data/processed_EV.csv`  
- `data/processed_NE.csv`  
- `data/processed_NP.csv`

**Outputs:**  
- `Outputs/topics_2.csv`  
- `Outputs/topic_centroids.csv`  
- `Outputs/topic_top_words.csv`  
- `Outputs/topic_labels_from_my_reps.csv`


In [ ]:
# --- Reproducibility Settings ---
import random, numpy as np, torch
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


In [ ]:
import os
import re
import copy
import pandas as pd
import matplotlib.pyplot as plt

import umap
import hdbscan
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import MaximalMarginalRelevance

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from bertopic.dimensionality import BaseDimensionalityReduction
from sklearn.cluster import KMeans



In [ ]:
# Load datasets
ev_csv = pd.read_csv("../data/processed_EV.csv")
ne_csv = pd.read_csv("../data/processed_NE.csv")
np_csv = pd.read_csv("../data/processed_NP.csv")

ev_csv["section"] = "EV"
ne_csv["section"] = "NE"
np_csv["section"] = "NP"

df = pd.concat([ev_csv, ne_csv, np_csv], ignore_index=True)

In [ ]:
# Standardize ChatGPT spellings
df['text'] = df['text'].str.replace(r'\bChat GPT\b', 'ChatGPT', regex=True)
df['text'] = df['text'].str.replace(r'\b GPT\b', 'ChatGPT', regex=True)
df['text'] = df['text'].str.replace(r'\b GTP\b', 'ChatGPT', regex=True)

# Clean function
def clean_text(text):
    text = re.sub(r"\n|\t|\xa0", " ", str(text))
    text = re.sub(r"Listen \d+:\d+\s?", "", text)
    text = re.sub(r"\b\d+,", lambda x: x.group().replace(",", ""), text)
    contractions_pattern = re.compile(r"(\b\w+?(?:'[dtlm]|n't|'s|'ve|'re)\b)", re.IGNORECASE)
    contractions_dict = {
        "won't": "will not", "can't": "cannot", "n't": " not",
        "'s": " is", "'d": " would", "'ll": " will", "'m": " am",
        "'ve": " have", "'re": " are"
    }
    def replace_contractions(match):
        return contractions_dict.get(match.group(0).lower(), match.group(0))
    text = contractions_pattern.sub(replace_contractions, text)
    return re.sub(r"\s+", " ", text).strip()

# Clean dataset
df_copy = df.copy()
df_copy['text'] = df_copy['text'].apply(lambda x: clean_text(x))
#docs = df_clean['text'].tolist()


In [ ]:
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

class SemanticChunker:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model_name)

    def chunk_text(self, text, context_window=1, percentile_threshold=95, min_chunk_size=2):
        """Chunk a transcript string into semantically coherent segments"""
        # 1. Sentence tokenize
        sentences = sent_tokenize(text)

        # 2. Add context window
        contextualized = []
        for i in range(len(sentences)):
            start = max(0, i - context_window)
            end = min(len(sentences), i + context_window + 1)
            context = " ".join(sentences[start:end])
            contextualized.append(context)

        # 3. Encode sentences
        embeddings = self.model.encode(contextualized)

        # 4. Compute semantic distances
        distances = [
            1 - cosine_similarity([embeddings[i]], [embeddings[i+1]])[0][0]
            for i in range(len(embeddings)-1)
        ]

        # 5. Identify breakpoints
        threshold = np.percentile(distances, percentile_threshold)
        breakpoints = [i for i, dist in enumerate(distances) if dist > threshold]

        # 6. Create chunks
        chunks, start_idx = [], 0
        for bp in breakpoints:
            chunks.append(" ".join(sentences[start_idx:bp+1]))
            start_idx = bp+1
        chunks.append(" ".join(sentences[start_idx:]))

        # 7. Merge small chunks
        final_chunks = []
        for chunk in chunks:
            if len(chunk.split()) < min_chunk_size and final_chunks:
                final_chunks[-1] += " " + chunk
            else:
                final_chunks.append(chunk)

        return final_chunks


In [ ]:
chunker = SemanticChunker("all-MiniLM-L6-v2")

all_chunks = []

# Loop through each row to keep metadata
for idx, row in df_copy.iterrows():
    speaker = row['speaker']
    section = row['section']
    text = row['text']

    chunks = chunker.chunk_text(
        text,
        context_window=1,
        percentile_threshold=95,
        min_chunk_size=3
    )

    # Store with metadata
    for chunk in chunks:
        all_chunks.append({
            "orig_doc_id": idx,      # the original df_copy index
            "speaker": speaker,
            "section": section,
            "sentence": chunk
        })

print(f"✅ Total semantic chunks: {len(all_chunks)}")

# Convert to DataFrame
df_chunks = pd.DataFrame(all_chunks)


#export to csv
df_chunks.to_csv("../Outputs/semantic_chunks_docids.csv",  index=True)

In [ ]:
# # Use chunker instead of raw docs
# chunker = SemanticChunker("all-MiniLM-L6-v2")

# all_chunks = []
# for text in df_copy['text']:
#     all_chunks.extend(chunker.chunk_text(text, context_window=1, percentile_threshold=95, min_chunk_size=3))

# print(f"Total semantic chunks: {len(all_chunks)}")


docs = df_chunks.sentence


In [ ]:
df_clean = pd.DataFrame({
    "doc_id": range(len(docs)),
    "sentence": docs
})

In [ ]:
# Embeddings

model_prep = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model_prep.encode(docs, batch_size=32, show_progress_bar=False)


In [ ]:
# UMAP (Dimentionality Reduction)

reducer_prep = umap.UMAP(
    n_neighbors=15,
    n_components=2,
    metric='cosine',
    random_state=42,
    #min_dist=0.0,
    n_jobs=1
)
embedding_prep = reducer_prep.fit_transform(embeddings)
df_clean['prep_x_coord'] = embedding_prep[:, 0]
df_clean['prep_y_coord'] = embedding_prep[:, 1]


In [ ]:
# Clustering (HDBSCAN + Outlier Reassignment)

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=7,
    min_samples=5,
    metric='euclidean',
    cluster_selection_method='leaf',
    prediction_data=True
).fit(embedding_prep)

df_clean['prep_cluster_hdb_leaf'] = clusterer.labels_



In [ ]:
## BERTopic


#Tokenizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

custom_stopwords = ENGLISH_STOP_WORDS.union({
    "say","said","like","know","think","just","would","get","use","make",
    "really","also","lot","even","yeah","thing","see","much","well",
    "going","people","information","didn","mean","don","probably", "100",
    "little","bit","want","one","time","kind","maybe", "ve", "did", "ok", "00"
})

vectorizer_model = CountVectorizer(
    stop_words=list(custom_stopwords),
    ngram_range=(1,3), 
    min_df=4
)

#Topic Representation (c-TF-IDF)
ctfidf = ClassTfidfTransformer(reduce_frequent_words=True)

# ctfidf = ClassTfidfTransformer(
#     seed_words=["chatgpt", "copilot", "gemini"],
#     seed_multiplier=2
# )

#Finetuning (MMR)
representation_model = MaximalMarginalRelevance(diversity=0.75)

#Model Fit 
topic_model = BERTopic(
    embedding_model=model_prep,
    umap_model=reducer_prep,
    hdbscan_model=clusterer,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf,
    representation_model=representation_model,
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs, embeddings=embeddings)






In [ ]:
# Evaluate coherence & diversity
docs_info = topic_model.get_document_info(docs)
documents_per_topic = docs_info.groupby(['Topic'], as_index=False).agg({'Document': ' '.join})
cleaned_docs = topic_model._preprocess_text(documents_per_topic.Document.values)
analyzer = topic_model.vectorizer_model.build_analyzer()
tokens = [analyzer(doc) for doc in cleaned_docs]

topic_ids = [tid for tid in topic_model.get_topics().keys() if tid != -1]
bertopic_topics = [[w for (w, _) in (topic_model.get_topic(i) or [])[:10]] for i in topic_ids]

dictionary = Dictionary(tokens)
coherence_model = CoherenceModel(
    topics=bertopic_topics, texts=tokens, dictionary=dictionary,
    coherence='c_v', topn=10, processes=1
)
print("Coherence Score:", coherence_model.get_coherence())

def topic_diversity(topics_list, topk=10):
    words_flat = [word for topic in topics_list for word in topic[:topk]]
    return len(set(words_flat)) / len(words_flat) if words_flat else 0
print("Topic Diversity:", topic_diversity(bertopic_topics))

In [ ]:
# Visualizations (optional)
%matplotlib inline
from IPython.display import display, HTML
display(HTML("<style>.output_wrapper, .output {overflow:auto; max-width: 100% !important;}</style>"))


barchart = topic_model.visualize_barchart(top_n_topics=25, n_words=8, width=250, title="")
barchart.show()



In [ ]:
fig = topic_model.visualize_heatmap(
    top_n_topics=10,             
    n_clusters=None,
    width=1000,
    height=600
)
fig.show()

hierarchical_topics = topic_model.hierarchical_topics(docs)
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

### Outputs

In [ ]:
# Export topics.csv with no replacement
os.makedirs("../Outputs", exist_ok=True)
docs_info = topic_model.get_document_info(docs)
export_df = docs_info[['Topic', 'Probability']].copy()
export_df.insert(0, "doc_id", range(len(export_df)))
export_df.rename(columns={'Topic': 'topic_id', 'Probability': 'prob'}, inplace=True)
export_df.to_csv("../Outputs/topics_2.csv", index=False)
print("✅ Wrote Outputs/topics_2.csv")

In [ ]:
#get_topic_info()
topic_info = topic_model.get_topic_info()
topic_info

In [ ]:
#Topic assignment per sentence

assignments = topic_model.get_document_info(docs)
assignments = assignments[['Document', 'Topic', 'Probability']].copy()
assignments.insert(0, "doc_id", range(len(assignments)))
assignments.rename(columns={
    'Document': 'sentence',
    'Topic': 'topic_id',
    'Probability': 'prob'
}, inplace=True)
assignments.to_csv("../Outputs/topic_assignment.csv", index=False)
print("✅ Wrote Outputs/topic_assignment.csv")

In [ ]:
#Topic → Top Words 

topic_words = []
for topic_id in topic_model.get_topics().keys():
    if topic_id == -1:  # skip outliers
        continue
    words = [w for w, _ in topic_model.get_topic(topic_id)]
    topic_words.append({"topic_id": topic_id, "top_words": words})

topic_words_df = pd.DataFrame(topic_words)
topic_words_df.to_csv("../Outputs/topic_top_words.csv", index=False)
print("✅ Wrote Outputs/topic_top_words.csv")

In [ ]:
#Topic Centroids + Representative Docs + Avg Familiarity (Top-15 only) 

from sklearn.metrics.pairwise import cosine_similarity

topic_labels = assignments['topic_id'].values
centroids = {}

top_docs = []
k = 15   # number of representative docs per topic

for topic_id in np.unique(topic_labels):
    if topic_id == -1:   # skip outliers
        continue
    idx = np.where(topic_labels == topic_id)[0]
    topic_embeddings = embeddings[idx]
    centroid = topic_embeddings.mean(axis=0)
    centroids[topic_id] = centroid
    
    # Similarity of each doc to centroid
    sims = cosine_similarity([centroid], topic_embeddings)[0]
    
    # Select top-k docs by similarity
    top_idx = idx[np.argsort(sims)[::-1][:k]]
    top_sims = sims[np.argsort(sims)[::-1][:k]]  # similarities of top-k
    avg_topk_sim = top_sims.mean()               # avg similarity of top-k docs
    
    for rank, (doc_idx, sim) in enumerate(zip(top_idx, top_sims), 1):
        top_docs.append({
            "topic_id": topic_id,
            "rank": rank,
            "doc_id": doc_idx,
            "sentence": docs[doc_idx],
            "similarity": sim,
            "avg_top15_similarity": avg_topk_sim  # same for all top-15 in topic
        })

# --- Save combined results ---
top_docs_df = pd.DataFrame(top_docs)
top_docs_df.to_csv("../Outputs/topic_representative_docs.csv", index=False)
print("✅ Wrote Outputs/topic_representative_docs.csv")


#Topic centroids
centroid_df = pd.DataFrame.from_dict(centroids, orient="index")
centroid_df.index.name = "topic_id"
centroid_df.reset_index(inplace=True)
centroid_df.to_csv("../Outputs/topic_centroids.csv", index=False)
print("✅ Wrote Outputs/topic_centroids.csv")

In [ ]:
#Topic representative to Pivot


df_long = pd.read_csv("../Outputs/topic_representative_docs.csv")

# Pivot into wide format
df_wide = df_long.pivot(
    index="rank",        # rows will be ranks (1, 2, 3…)
    columns="topic_id",  # columns will be topics
    values="sentence"    # cell values are the representative sentences
)

# Reset column names for clarity
df_wide.columns = [f"Topic_{c}" for c in df_wide.columns]
df_wide.reset_index(inplace=True)

# Save to CSV
df_wide.to_csv("../Outputs/topic_representative_docs_transposed_centroid.csv", index=False)
print("✅ Wrote Outputs/topic_representative_docs_transposed_centroid.csv")

In [ ]:

#Probability Scores (Soft Assignments) 
# Matrix of probabilities for each doc across topics

prob_matrix = pd.DataFrame(probs, columns=[f"topic_{i}" for i in range(probs.shape[1])])
prob_matrix.insert(0, "doc_id", range(len(docs)))
prob_matrix.insert(1, "sentence", docs)
prob_matrix.to_csv("../Outputs/topic_probabilities.csv", index=False)
print("✅ Wrote Outputs/topic_probabilities.csv")

### Labeling

In [ ]:

from bertopic.representation import OpenAI
import openai

In [ ]:


# Load your existing topic info and representative docs
topic_info = topic_model.get_topic_info()              # has Topic + Name (keywords)
rep_docs = pd.read_csv("../Outputs/topic_representative_docs.csv")  # centroid-based reps

# OpenAI client
client = openai.OpenAI(api_key="")

# Prompt template

prompt = """
I have a topic that contains the following documents:
[DOCUMENTS]

The topic is described by the following keywords: [KEYWORDS].

Generate a short, distinct label (5–6 words) that captures what makes THIS topic unique 
compared to other topics in the dataset. My topics are all about AI, so ensure to make each topic unique and meaningful
Do not repeat the same phrasing (e.g., "AI Integration", "AI in Nursing") — instead, 
highlight unique angles such as ethics, teaching, Clinical Practice, Gnerative AI, Policies, Work & Documentation etc,.

Return only the label without quotes or formatting.
"""

# prompt = '''I have a topic described by the following keywords: [KEYWORDS]. 
# Based on the keywords, return only a short, human-readable label theme using not more than five words. 
# My topics are all about AI, so ensure to make each topic unique and meaningul
# Do not include quotes, asterisks, or extra formatting.'''

labels = []
for _, row in topic_info.iterrows():
    topic_id = row["Topic"]
    keywords = ", ".join(row["Representation"])

    # Skip outlier topic -1
    if topic_id == -1:
        continue

    # Get top 5 of your representative docs for this topic
    docs_text = "\n".join(
        rep_docs[rep_docs["topic_id"] == topic_id]
        .sort_values("rank")
        .head(5)["sentence"]
        .tolist()
    )

    # Replace placeholders
    final_prompt = prompt.replace("[KEYWORDS]", keywords).replace("[DOCUMENTS]", docs_text)

    # Call GPT
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": final_prompt}],
        temperature=0.1
    )

    label = response.choices[0].message.content.strip()

    labels.append({"topic_id": topic_id, "topic_label": label})

# Save results
labels_df = pd.DataFrame(labels)
labels_df.to_csv("../Outputs/topic_labels_from_my_reps.csv", index=False)
print(labels_df.head())
